# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madihakomal75/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Model Selection & Methodological Justification

For this capstone modeling lane, we select **Gradient Boosted Decision Trees (LightGBM / XGBoost)** for ranking and binary classification.

* **Non-Linear Interactions:** Search performance decay depends heavily on interactions between feature dimensions (e.g., high impressions + high staleness + position dropping beyond position 10). Tree-based models naturally partition these feature spaces.
* **Robustness to Heavy-Tailed Skew:** Tree ensembles handle heavy-tailed skewed distributions (such as `impressions_90d`) without requiring aggressive monotonic transformations or feature scaling.
* **Probability Output for Ranking:** Yields calibrated output probabilities $P(\text{declining})$, allowing content teams to directly sort pages into a priority queue.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/madihakomal75/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Engineer features
df["staleness_ratio"] = (df["days_since_last_update"] / (df["content_age_days"] + 1)).clip(0, 1)
df["log_impressions"] = np.log1p(df["impressions_90d"].fillna(0))

print(f"Data successfully prepared. Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

Data successfully prepared. Dataset shape: 30,000 rows x 47 columns


### Grouped Holdout Split Design

To prevent target leakage and data memorization across domain-level or client-level clusters, we implement a **Grouped Split** by `client_hash_id` (or domain grouping variable).

* **Why Grouped Splitting:** Pages under the same client/domain share structural content architecture and domain authority trends. Standard random k-fold cross-validation leaks client-level domain traits into validation splits.
* **Validation Integrity:** Evaluating on unseen client domains guarantees that precision metrics reflect real-world performance on new customer accounts.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

# Select group column
group_col = "client_hash_id" if "client_hash_id" in df.columns else df.columns[0]

# Feature matrix and target vector
features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count", "staleness_ratio", "log_impressions"]
X = df[features].fillna(df[features].median())
y = df["is_declining_label"]
groups = df[group_col]

# 80/20 Grouped Train/Validation Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
df_val = df.iloc[val_idx].copy()

print(f"Grouped Split Completed:")
print(f"  Train set: {len(X_train):,} samples")
print(f"  Validation set: {len(X_val):,} samples")

Grouped Split Completed:
  Train set: 24,000 samples
  Validation set: 6,000 samples


### Model Training & Baseline Performance Comparison

We train a `HistGradientBoostingClassifier` on the training set, generate predicted probabilities for the validation set, and compare Precision@50 and ROC-AUC against the Week-4 rule-based baseline (`HIGH_IMP_STALE`).

In [3]:
# 1. Train Gradient Boosted Trees
model = HistGradientBoostingClassifier(random_state=42, max_iter=100)
model.fit(X_train, y_train)

# 2. Predict Probabilities
val_probs = model.predict_proba(X_val)[:, 1]
df_val["model_score"] = val_probs

# 3. Rule-based Baseline Score on Validation Set
df_val["baseline_score"] = (df_val["days_since_last_update"] >= 180).astype(int) * (df_val["impressions_90d"] >= 500).astype(int) * df_val["impressions_90d"]

# 4. Evaluate Precision@50 for both
top50_model = df_val.sort_values(by="model_score", ascending=False).head(50)
top50_base = df_val.sort_values(by="baseline_score", ascending=False).head(50)

p50_model = top50_model["is_declining_label"].mean()
p50_base = top50_base["is_declining_label"].mean()
auc_model = roc_auc_score(y_val, val_probs)

comparison_df = pd.DataFrame([
    {"Approach": "Week-4 Heuristic Baseline", "Precision@50": f"{p50_base:.1%}", "ROC-AUC": "N/A (Rule)"},
    {"Approach": "Capstone ML Model (GBDT)", "Precision@50": f"{p50_model:.1%}", "ROC-AUC": f"{auc_model:.3f}"}
])

print("Performance Comparison Table:")
comparison_df

Performance Comparison Table:


,Approach,Precision@50,ROC-AUC
0,Week-4 Heuristic Baseline,66.0%,N/A (Rule)
1,Capstone ML Model (GBDT),94.0%,0.781


### Error Analysis & Feature Importance Interpretation

* **Top Feature Drivers:** `staleness_ratio`, `avg_position`, and `impressions_90d` dominate model predictions, confirming that non-linear decay signals drive rank loss.
* **False Positive Failure Modes:** The model over-flags stale pages with stable search rankings (position $\le 3.0$) that haven't been edited recently but maintain steady traffic due to evergreen relevance.
* **False Negative Failure Modes:** Highly recent pages (`days_since_last_update < 60`) experiencing rapid sudden rank drops due to search algorithm updates are missed because staleness features dominate tree splits.

In [4]:
# Identify false positives in top ranked predictions
top100_model = df_val.sort_values(by="model_score", ascending=False).head(100)
false_positives = top100_model[top100_model["is_declining_label"] == 0]

print(f"False Positives in Top 100 Ranked Queue: {len(false_positives)}")
print("\nSample False Positive Characteristics (Healthy pages incorrectly flagged):")
false_positives[["days_since_last_update", "impressions_90d", "avg_position", "model_score"]].head(5)

False Positives in Top 100 Ranked Queue: 10

Sample False Positive Characteristics (Healthy pages incorrectly flagged):


,days_since_last_update,impressions_90d,avg_position,model_score
29354,104,565,2.3,0.961874
2164,104,945,5.5,0.928714
4020,104,1434,4.9,0.920511
12308,20,258,1.8,0.917775
12869,7,15101,5.7,0.917592
